# LASSER on Kaggle

GEARS with an optional GSR-style learned co-expression graph. Needs **Internet on** (GEARS downloads Norman and the GO graph from Harvard Dataverse) and a **GPU (T4)**.

Cells: install, get the code, run the tests (before any real training), baseline run, graph-learning run, results.

In [1]:
# 1. Dependencies. GEARS is vendored inside lasser/ (lasser/gears), so no cell-gears install.
#    Kaggle already has torch, numpy, pandas, scipy, scikit-learn, networkx, tqdm, requests.
!pip install -q torch_geometric scanpy pytest

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00


In [2]:
# 2. Get the code: clone, then move the self-contained package next to the notebook.
REPO_URL = "https://github.com/LegendaryDarkKnight/LASSER"   # <- set this
REPO_DIR = "/kaggle/working/lasser-repo"
!rm -rf {REPO_DIR} /kaggle/working/lasser
!git clone -q {REPO_URL} {REPO_DIR}
!mv {REPO_DIR}/lasser /kaggle/working/lasser
!git -C {REPO_DIR} log -1 --oneline

ab75303 (HEAD -> main, origin/main, origin/HEAD) Docs: vendored GEARS, dependencies and independence rules


In [3]:
# 3. Tests 1-8 (tiny subsets; first run also downloads Norman, ~10-20 min in total).
%env LASSER_DATA_DIR=/kaggle/working/data
%env LASSER_TEST_DIR=/kaggle/working/lasser_tests
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests -m "not smoke" -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/test_output.txt | grep -vE "it/s\]|s/it\]"

env: LASSER_DATA_DIR=/kaggle/working/data
env: LASSER_TEST_DIR=/kaggle/working/lasser_tests
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working
plugins: fast-array-utils-1.5.1, zarr-3.4.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 15 items

lasser-repo/tests/test_00_packaging.py EEE
lasser-repo/tests/test_01_parity.py Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!
here1
FLocal copy of split is detected. Loading...
Simul

In [4]:
# 3b. Optional test 9: full Norman, 1 epoch, flag off and on, checks the T4 memory budget.
%env LASSER_SMOKE=1
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests/test_09_smoke.py -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/smoke_output.txt | grep -vE "it/s\]|s/it\]"
%env LASSER_SMOKE=0

env: LASSER_SMOKE=1
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working
plugins: fast-array-utils-1.5.1, zarr-3.4.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 2 items

lasser-repo/tests/test_09_smoke.py Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
2026-10-06 08:32:56,112 INFO lasser.tracking: run eda6479b -> /kaggle/working/lasser_tests/runs/norman_seed1_base_20261006-083256_smoke
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!
here1
2026-10-06 08:32:56,189 INFO la

In [5]:
# 4. Load the data once and reuse it for both runs.
from lasser import GSRConfig, LasserConfig, load_pert_data, load_run, run_experiment

common = dict(dataset="norman", seed=1, epochs=1,
              data_dir="/kaggle/working/data", out_dir="/kaggle/working/runs",
              cache_dir="/kaggle/working/cache", repo_dir=REPO_DIR)
pert_data = load_pert_data(LasserConfig(**common))

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!


In [6]:
# 5. Baseline: graph_learning=False is plain GEARS (plus logging and saving).
base = run_experiment(LasserConfig(**common, graph_learning=False), pert_data)
base["run_dir"]

2026-10-06 08:34:25,925 INFO lasser.tracking: run cfb8a822 -> /kaggle/working/runs/norman_seed1_base_20261006-083425


Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!


here1
2026-10-06 08:34:26,002 INFO lasser.tracking: split hash f93e7d59ee239960 (copied ['norman_simulation_1_0.75_subgroup.pkl', 'norman_simulation_1_0.75.pkl'])
2026-10-06 08:34:26,004 INFO lasser.utils: [gears_init] start
2026-10-06 08:34:26,117 INFO lasser.utils: [gears_init] done in 0.1s, peak CUDA memory 0.00 GB
2026-10-06 08:34:26,118 ERROR lasser.run: run cfb8a822 failed
Traceback (most recent call last):
  File "/kaggle/working/lasser/run.py", line 163, in run_experiment
    ctx = prepare(cfg, pert_data, tracker)
  File "/kaggle/working/lasser/run.py", line 139, in prepare
    gears_obj = GEARS(pert_data, device=cfg.device)
  File "/kaggle/working/lasser/gears/gears.py", line 87, in __init__
    np.mean(self.adata.X[self.adata.obs.condition == 'ctrl'],
            ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/scipy/sparse/_index.py", line 30, in __getitem__
    index, new_shape = self._validate_indices(key)
                   

AttributeError: 'Series' object has no attribute 'nonzero'

In [ ]:
# 6. Graph learning: GSR-refined co-expression graph (defaults from the plan, section 11).
gl = run_experiment(LasserConfig(**common, graph_learning=True, gsr=GSRConfig()), pert_data)
gl["run_dir"], gl["checks"]["h2"], gl["checks"]["refine"]

In [ ]:
# 7. Results: the global run table, and any past run loaded back.
import pandas as pd
pd.set_option("display.max_columns", 200)
runs = pd.read_csv("/kaggle/working/runs/runs.csv")
cols = ["run_id", "seed", "graph_learning", "gsr_init_gene_emb", "graph_num_edges", "graph_isolated_genes",
        "test_pearson_delta", "test_mse_top20_de", "test_nmse_top20_de",
        "test_unseen_single_pearson_delta", "test_combo_seen0_pearson_delta",
        "test_combo_seen1_pearson_delta", "test_combo_seen2_pearson_delta", "run_dir"]
display(runs[[c for c in cols if c in runs.columns]])

r = load_run(gl["run_dir"])
display(r["epoch_log"])
r["test_metrics"]["paper"]["subgroups"]